In [15]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset, DataLoader
drive.mount('/content/drive')
import cv2
from google.colab.patches import cv2_imshow
from torch.utils.data import DataLoader
import numpy as np
import torchvision.transforms as transforms
import timm
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
ROOT = "/content/drive/MyDrive/Digital_Knee_X_ray_Images"
TRAIN_DATASET_DIR = "MedicalExpert-I"
VALIDATE_DATASET_DIR = "MedicalExpert-II"

In [5]:
class KneeXRayDataset(Dataset):

    def load_images_path(self):
        #iter each category
        # print(self.categories)
        for i, category in enumerate(self.categories):
            category_path = os.path.join(self.root, category)

            #iter file in category
            for file_path in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, file_path))
                self.labels.append(i)

    def load_image_from_path(self,imagePath):
        return cv2.imread(imagePath)
        
    def __init__(self, root, train_dataset_dir, validate_dataset_dir, transform , train=True, ):
        self.root = root
        self.transform  = transform

        # determine which dir will use (train , val)
        if train :
            self.root = os.path.join(root, train_dataset_dir)
        else:
            self.root = os.path.join(root,validate_dataset_dir)

        # get all category (0 -> 4)
        self.categories = os.listdir(self.root)
        self.image_paths = []
        self.labels = []

        # load images from dataset
        self.load_images_path()

        
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        image = self.load_image_from_path(self.image_paths[idx])
        # cv2_imshow(image)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()
        return self.transform(image), self.labels[idx]



In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224))    
])

In [7]:
train_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, transform=transform)
validate_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, train=False , transform=transform)

In [8]:
train_dataset_loader =  DataLoader(train_dataset, batch_size=16, shuffle=True , drop_last=False)
validate_dataset_loader = DataLoader(validate_dataset, batch_size=16, drop_last=False)

In [10]:
images, label = next(iter(train_dataset_loader))
# images = np.transpose(images, (2,0,1))
# cv2_imshow(images.numpy())
# cv2.waitKey(0)
print(images.shape)
print(label)

# for images,labels in train_dataset:
#     print(images.shape)
#     print(labels)

torch.Size([16, 3, 224, 224])
tensor([1, 0, 1, 1, 3, 3, 0, 1, 4, 0, 4, 2, 4, 2, 0, 0])


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [25]:
#define transfer learning model function
def create_transfer_learning_model(model_name: str="efficientnet_b0", pretrained: bool = True, num_classes:int = 5, device = device):
    model = timm.create_model(model_name, pretrained= pretrained, num_classes=5)

    #frezze all other layer
    for param in model.parameters():
        param.requires_grad = False

    #unfrezze the class classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model.to(device)


efficientnet_b0_model = create_transfer_learning_model()

In [26]:
# loss function & optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, efficientnet_b0_model.parameters()), lr=1e-3)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    # model is tranining mode
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc=" Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # delete previous grad
        optimizer.zero_grad()
        outputs = model(images)

        # calc loss
        loss = criterion(outputs, labels)

        # backward to calc loss
        loss.backward()

        # optimize model
        optimizer.step()

        # result
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total

    

    

In [36]:
def validate(model, loader, criterion, device):

    # eval mode
    model.eval()

    # init
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=" Validating", leave=False):

            #load to device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total 

    

In [ ]:
EPOCHS = 10
best_val_acc = 0.0


for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(efficientnet_b0_model, train_dataset_loader, criterion, optimizer, device)

    print(f"  -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    val_loss, val_acc = validate(efficientnet_b0_model, validate_dataset_loader, criterion, device)
    print(f"  -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        torch.save(efficientnet_b0_model.state_dict(), 'best_efficientnet_b0_transfer.pth')

        print(f"  [Save] Best val acc: {best_val_acc:.2f}%")
        
